# Step 0 checks — anchor weight direction & per-zone duration
Run from the `notebooks/` folder or the project root. Needs the dataset in `data/`.
Run top to bottom. **Nothing here changes your pipeline or results** — it only reads data and prints.

## Setup — identical loading to `run_custom_knn_cv.py`

In [1]:
import os, sys, pickle
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, recall_score

# Work from the project root, exactly like the script does
ROOT = Path.cwd().parent if Path.cwd().name in ("experiments", "notebooks") else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)

from config.settings import RAW_PATH, GQCM_PATH, DATA_PATH, ZQCM_PATH, GQCM_Hermite_path
from src.data_loader import load_data
from src.weights import compute_weight_matrix, get_train_weights
from src.knn_custom import compute_weighted_MD, get_top_k, compute_probabilities

X_raw, X_zqcm, X_gqcm, X, y, X_gqcm_hermite = load_data(
    RAW_PATH, ZQCM_PATH, GQCM_PATH, DATA_PATH, GQCM_Hermite_path
)

FOLD_PATH = ROOT / "config" / "fold_indices.pkl"
with open(FOLD_PATH, "rb") as f:
    folds = pickle.load(f)            # list of (train_idx, test_idx)

print("X_raw", X_raw.shape, "| X_gqcm", X_gqcm.shape, "| X (NQCM)", X.shape)
zones, counts = np.unique(y, return_counts=True)
print("zones:", dict(zip(zones, counts)))
print("number of folds:", len(folds))

Project root: <project root>
X_raw (5400, 4) | X_gqcm (5400, 4) | X (NQCM) (5400, 4)
zones: {np.int64(1): np.int64(1420), np.int64(2): np.int64(1375), np.int64(3): np.int64(1128), np.int64(4): np.int64(1477)}
number of folds: 5


## Test 1 — Where do the errors actually sit? (full dataset)
Zero counts (raw) and outlier counts (GQCM NaN) per zone × anchor, then the resulting weight matrix.
**Check:** in Zone 4 (NLOS corridor), the two NLOS anchors should have the most errors — and therefore the largest weights.

In [2]:
anchors = [f"A{j+1}" for j in range(X_raw.shape[1])]
zero_ct = pd.DataFrame([(X_raw[y == z] == 0).sum(axis=0) for z in zones], index=zones, columns=anchors)
nan_ct  = pd.DataFrame([np.isnan(X_gqcm[y == z]).sum(axis=0) for z in zones], index=zones, columns=anchors)
print("Zeros in RAW (zone x anchor):");        display(zero_ct)
print("NaN outliers in GQCM (zone x anchor):"); display(nan_ct)

W_full = compute_weight_matrix(X_raw, X_gqcm, y)
print("Weight matrix W (rows = zones, cols = anchors):")
display(pd.DataFrame(W_full, index=zones, columns=anchors).round(3))

Zeros in RAW (zone x anchor):


,A1,A2,A3,A4
1,0,0,237,0
2,0,0,0,0
3,0,0,0,0
4,0,59,2,0


NaN outliers in GQCM (zone x anchor):


,A1,A2,A3,A4
1,46,49,26,63
2,66,49,46,75
3,35,32,70,10
4,77,78,45,6


Weight matrix W (rows = zones, cols = anchors):


,A1,A2,A3,A4
1,0.109,0.116,0.625,0.150
2,0.280,0.208,0.195,0.318
3,0.238,0.218,0.476,0.068
4,0.288,0.513,0.176,0.022


## Test 2 — Per-fold weight matrices: zero rows / zero entries?
Same computation as the script (training rows only).
- A row summing to **0** → that zone's training samples get distance 0 to every test point (a real bug).
- A single **0 entry** → that anchor is ignored entirely for that zone.

In [3]:
fold_W = []
for i, (tr, te) in enumerate(folds):
    W = compute_weight_matrix(X_raw[tr], X_gqcm[tr], y[tr])
    fold_W.append(W)
    print(f"Fold {i+1}: row sums = {W.sum(axis=1).round(3)} | "
          f"zero entries (zone_idx, anchor_idx) = {np.argwhere(W == 0).tolist()}")

Fold 1: row sums = [1. 1. 1. 1.] | zero entries (zone_idx, anchor_idx) = []
Fold 2: row sums = [1. 1. 1. 1.] | zero entries (zone_idx, anchor_idx) = []
Fold 3: row sums = [1. 1. 1. 1.] | zero entries (zone_idx, anchor_idx) = []
Fold 4: row sums = [1. 1. 1. 1.] | zero entries (zone_idx, anchor_idx) = []
Fold 5: row sums = [1. 1. 1. 1.] | zero entries (zone_idx, anchor_idx) = []


## Test 3 — Direction test on a toy example
Same absolute difference on anchor A1; only A1's weight changes.
If the distance grows with A1's weight → **a higher weight = more influence** on neighbour selection.

In [4]:
X_train_toy = np.array([[0.0, 0.0, 0.0, 0.0]])
X_test_toy  = np.array([[1.0, 0.0, 0.0, 0.0]])   # differs only on A1

for w1 in [0.1, 0.4, 0.7]:
    rest = (1 - w1) / 3
    w = np.array([[w1, rest, rest, rest]])
    d = compute_weighted_MD(X_test_toy, X_train_toy, w)
    print(f"weight on A1 = {w1:.1f}  ->  distance = {d[0,0]:.3f}")

weight on A1 = 0.1  ->  distance = 0.100
weight on A1 = 0.4  ->  distance = 0.400
weight on A1 = 0.7  ->  distance = 0.700


## Test 4 — Direction on real data: which anchor drives the distance?
For one Zone 4 test sample, split its distance to the nearest Zone 4 training sample into per-anchor contributions.

In [5]:
fold_no = 4                          # index 4 = thesis Fold 5
tr, te = folds[fold_no]
W = fold_W[fold_no]

# Zone 4 = the label with 1,477 samples
z4 = zones[counts == 1477][0] if np.any(counts == 1477) else zones[-1]
print("Zone 4 label:", z4)

t_idx = te[y[te] == z4][0]
cand  = tr[y[tr] == z4]
d_all = compute_weighted_MD(X[[t_idx]], X[cand], get_train_weights(W, y[cand]))[0]
nn = cand[np.argmin(d_all)]

zi = list(zones).index(z4)
abs_diff = np.abs(X[t_idx] - X[nn])
contrib  = abs_diff * W[zi]
display(pd.DataFrame({"abs diff": abs_diff, "weight": W[zi], "contribution": contrib,
                      "share %": 100 * contrib / contrib.sum()}, index=anchors).round(3))

Zone 4 label: 4


,abs diff,weight,contribution,share %
A1,14.150,0.281,3.980,29.173
A2,15.623,0.509,7.951,58.285
A3,7.824,0.183,1.432,10.498
A4,10.413,0.027,0.279,2.045


## Test 5 (exploratory, optional) — uniform vs literal vs inverted weights
Uses **your own** classifier functions, same k=3 and folds as the thesis.
**Validation first:** the *literal* rows must reproduce your thesis numbers
(mean accuracy ≈ 0.9946; Fold 5 Zone 1 recall = 1.0000). If they don't, stop and tell me.

In [6]:
def predict(X_te, X_tr, y_tr, tw, k=3):
    D = compute_weighted_MD(X_te, X_tr, tw)
    sorted_idx = np.argsort(D, axis=1)
    labels, dists = get_top_k(sorted_idx, D, y_tr, k)
    P, uz = compute_probabilities(labels, dists)
    return uz[np.argmax(P, axis=1)]

def weight_variants(W):
    uniform  = np.full_like(W, 1 / W.shape[1])
    inv      = 1 - W
    inverted = inv / inv.sum(axis=1, keepdims=True)
    return {"uniform": uniform, "literal (Eq. 20)": W, "inverted": inverted}

rows = []
for fold_no, (tr, te) in enumerate(folds):
    for name, Wv in weight_variants(fold_W[fold_no]).items():
        p = predict(X[te], X[tr], y[tr], get_train_weights(Wv, y[tr]))
        rec = recall_score(y[te], p, labels=zones, average=None, zero_division=0)
        rows.append({"fold": fold_no + 1, "weights": name,
                     "accuracy": accuracy_score(y[te], p),
                     **{f"recall {z}": r for z, r in zip(zones, rec)}})

res = pd.DataFrame(rows)
display(res.round(4))
display(res.groupby("weights")[["accuracy"]].agg(["mean", "std"]).round(4))

,fold,weights,accuracy,recall 1,recall 2,recall 3,recall 4
0,1,uniform,1.0000,1.0000,1.0000,1.0000,1.0
1,1,literal (Eq. 20),1.0000,1.0000,1.0000,1.0000,1.0
2,1,inverted,1.0000,1.0000,1.0000,1.0000,1.0
3,2,uniform,1.0000,1.0000,1.0000,1.0000,1.0
4,2,literal (Eq. 20),1.0000,1.0000,1.0000,1.0000,1.0
5,2,inverted,1.0000,1.0000,1.0000,1.0000,1.0
6,3,uniform,1.0000,1.0000,1.0000,1.0000,1.0
7,3,literal (Eq. 20),0.9907,1.0000,0.9636,1.0000,1.0
8,3,inverted,1.0000,1.0000,1.0000,1.0000,1.0
9,4,uniform,1.0000,1.0000,1.0000,1.0000,1.0


accuracy        
                     mean     std
weights                          
inverted           0.9754  0.0551
literal (Eq. 20)   0.9946  0.0079
uniform            0.9944  0.0124

## Test 6 — 20 s or 80 s per zone? (Check 2)
First look at the raw file's columns and first rows to find the time column.

In [ ]:
raw_df = pd.read_excel(RAW_PATH)
print(raw_df.shape)
print(raw_df.columns.tolist())
display(raw_df.head())

In [ ]:
TIME_COL = "timestamp"   # TODO: set to the time column name printed above
ZONE_COL = "zone_label"

t = pd.to_datetime(raw_df[TIME_COL])
# If the time values are big numbers (e.g. 1710411725.1): pd.to_datetime(raw_df[TIME_COL], unit="s")
# If they are seconds from start (0.00, 0.05, ...): tell Claude, the code below needs a small change
print("median dt between samples:", t.diff().dt.total_seconds().median(), "s")
for z, g in raw_df.groupby(ZONE_COL, sort=False):
    dur = (t[g.index].max() - t[g.index].min()).total_seconds()
    print(f"Zone {z}: {len(g)} samples, {dur:.1f} s, {len(g)/dur:.1f} Hz")